# SceneSeg

## I. Getting Started Guide

### 1. SceneSeg Model Introduction

Welcome to the first model in Autoware's VisionPilot (previously known as Privately-Owned Vehicle project) - SceneSeg.

SceneSeg is a semantic segmentation network in the VisionPilot suite designed for full-scene understanding in
urban driving environments. It follows an Encoder-Context-Decoder architecture to balance broad semantic awareness
with fine pixel-level boundary precision.

The architecture components include:

- A pretrained EfficientNet-B0 Backbone (Encoder) that extracts hierarchical multi-scale features and returns
feature maps from shallow to deep stages for skip-link fusion.

- A Scene Context module that applies global average pooling on the deepest feature map, passes it through MLP
layers, reshapes it to a spatial prior ($10 \times 20$), and expands it back with convolution layers.

- A residual attention mechanism in the context stage ($Context \times Features + Features$) that injects global
scene cues while preserving original encoded detail.

- A Scene Neck (Decoder) with three upsampling blocks and skip connections from intermediate encoder features
to recover spatial structure at progressively higher resolutions.

- A SceneSeg Head with two additional upsampling/convolution blocks and a final 3-channel prediction layer
for pixel-wise scene segmentation output.

As a real-time scene understanding model, SceneSeg processes front-view camera frames and predicts three semantic
classes: **background objects**, **foreground dynamic objects**, and **road**. The model is trained with diverse
driving datasets (including ACDC, BDD100K, IDDAW, MUSES, Mapillary, and Comma10K) to improve robustness across
weather, lighting, and road-domain variations.

![](../../Media/SceneSeg_GIF.gif)

### 2. Environment Setup

**If you have done these steps of environment setup in other End-to-End Model Tutorials, please skip this and move forward to the next subsection `3. Download Models`.**

First, we need to clone the repository to your local environment. We will use the official repository from the
Autoware Foundation.

In [ ]:
import os

base_dir = os.getcwd()

# Clone Autoware VisionPilot repo, if not yet done
if not os.path.exists("./autoware_vision_pilot"):
    !git clone https://github.com/autowarefoundation/autoware_vision_pilot.git

The VisionPilot project relies on several key libraries, including **PyTorch** for model execution and
**OpenCV** for image processing.

We will use `pip` to install the required dependencies directly from the `requirements.txt` file provided in 
the `Models` directory.

In [ ]:
# Install required dependencies
%pip install --upgrade pip
%pip install -r autoware_vision_pilot/Models/requirements.txt

# Verify the installation (checking torch as a primary dependency)
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA availability: {torch.cuda.is_available()}")

### 3. Download Models

You can manually download the model weight files, using the links below. Subsequent tutorial sections assume
you already downloaded them and put them inside directory: 
`autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to Download Pytorch Model Weights (*.pth)](https://drive.google.com/file/d/1vCZMdtd8ZbSyHn1LCZrbNKMK7PQvJHxj/view?usp=sharing)
- [Link to Download Traced Pytorch Model (*.pt)](https://drive.google.com/file/d/1G2pKrjEGLGY1ouQdNPh11N-5LlmDI7ES/view?usp=drive_link)
- [Link to Download ONNX FP32 Weights (*.onnx)](https://drive.google.com/file/d/1l-dniunvYyFKvLD7k16Png3AsVTuMl9f/view?usp=drive_link)
- [Link to Download ONNX INT8 Weights (*.onnx)](https://drive.google.com/file/d/1yAIsUukWKBsbIvoKVAabwwKPmSXRy24D/view?usp=drive_link)

You can also use below script block, which uses `gdown` to download above model weights into such default directory.

In [ ]:
%pip install gdown

import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1vCZMdtd8ZbSyHn1LCZrbNKMK7PQvJHxj",
        "filepath"  : os.path.join(base_dir, model_dir, "sceneseg.pth")
    },
    "pt"    : {
        "url"       : "https://drive.google.com/uc?id=1G2pKrjEGLGY1ouQdNPh11N-5LlmDI7ES",
        "filepath"  : os.path.join(base_dir, model_dir, "sceneseg.pt")
    },
    "fp32"  : {
        "url"       : "https://drive.google.com/uc?id=1l-dniunvYyFKvLD7k16Png3AsVTuMl9f",
        "filepath"  : os.path.join(base_dir, model_dir, "sceneseg_fp32.onnx")
    },
    "int8"  : {
        "url"       : "https://drive.google.com/uc?id=1yAIsUukWKBsbIvoKVAabwwKPmSXRy24D",
        "filepath"  : os.path.join(base_dir, model_dir, "sceneseg_int8.onnx")
    }
}

for weight_type in metadata:

    url         = metadata[weight_type]["url"]
    filepath    = metadata[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading SceneSeg {weight_type} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"SceneSeg {weight_type} weights already exist at {filepath}. Skipping download.")

## II. Quick Test/Inference

Pre-requisites: please ensure that you have completed the above **Environment Setup** and **Download Models** first.

### 1. Image Inference

The `image_visualization.py` script facilitates batch processing of images by loading a pre-trained model checkpoint,
performing a forward pass on each image, and overlaying SceneSeg predictions onto the original frames.

#### a. Run Batch Image Inference

To run this in any environment, we must navigate to the specific visualization directory to ensure the
script's internal relative paths resolve correctly.

Key Script Parameters:
- `-p / --model_checkpoint_path`: location of your `.pth` file containing the trained weights.
- `-i / --input_image_dirpath`: directory containing `.png`, `.jpg`, or `.jpeg` files you wish to process.
- `-o / --output_image_dirpath`: destination where the script will save the visualization files
(naming them as `[original_id]_data.png`).

In [ ]:
# 1. Path declaration
# Here we use the previously downloaded Pytorch weight file.
# For input and output directories, you can change them to your preferred paths.
# For now we will use the default input and output directories provided in the repository.
CHECKPOINT = metadata["pth"]["filepath"]
INPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/sceneseg/images/"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/SceneSeg && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_DIR)} \
    -o {os.path.abspath(OUTPUT_DIR)}

#### b. Display Results in Notebook

After running the inference, we can use matplotlib and PIL to visualize the results directly within this notebook.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Path to the newly created results
result_files = sorted([
    f for f in os.listdir(OUTPUT_DIR)
    if f.endswith(".png")
])

if (result_files):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files)):
            img_path = os.path.join(OUTPUT_DIR, result_files[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Result: {result_files[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 2. Video Inference

In this subsection, we will apply the SceneSeg model to video streams.
The `video_visualization.py` script processes video files frame-by-frame, performs semantic segmentation,
and compiles the results into a new annotated video file.

The video inference pipeline utilizes `OpenCV` for frame extraction and writes a blended overlay output
for each frame to ensure consistent visual output.

Key script parameters:

- `-p / --model_checkpoint_path`: path to the trained SceneSeg weights.
- `-i / --input_video_filepath`: path to the source video file.
- `-o / --output_video_filepath`: destination path for the visualized video.
- `-v / --vis`: optional flag to enable a pop-up window showing the frames as they are processed.

Technical details:

- The script resizes input frames to $(640, 320)$ for inference to match the model's requirements, then resizes
  both visualization and frame to $(1280, 720)$ for output video writing.
- Release of video pointers and writers are automatically handled upon completion or error.

In [ ]:
# 1. Path declaration
# Similarly as image inference above:
# - Pytorch weight file.
# - Input/Output paths can be changed to preferred.
# - Using defaults for now.
INPUT_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
 )
OUTPUT_PATH = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/sceneseg/videos/highway_normal_output.avi"
 )

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/SceneSeg && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT)} \
    -i {os.path.abspath(INPUT_PATH)} \
    -o {os.path.abspath(OUTPUT_PATH)}

## III. Model Training

### 1. Model Training on New Data

#### a. Load pre-trained model or load vanilla un-trained model for training from scratch

In this repository, model loading behavior is controlled by `SceneSegTrainer(checkpoint_path=...)` in
`Models/training/scene_seg_trainer.py`, and instantiated from `Models/training/train_scene_seg.py`.

- If `checkpoint_path` is an empty string (`""`), the trainer initializes a vanilla `SceneSegNetwork`
  and trains from scratch.
- If `checkpoint_path` points to a `.pth` file, trainer loads those weights before training continues.

Current default in `train_scene_seg.py` is training from scratch:

```python
# Current code in Models/training/train_scene_seg.py
trainer = SceneSegTrainer()
```

To support both modes explicitly, use this pattern in `train_scene_seg.py`:

```python
# Add near argparse definitions
parser.add_argument(
    "-c", "--checkpoint_path",
    dest="checkpoint_path",
    default="",
    help="optional path to .pth checkpoint for resume/fine-tune"
 )

# After parsing args
checkpoint_path = args.checkpoint_path

if (checkpoint_path != ""):
    trainer = SceneSegTrainer(checkpoint_path = checkpoint_path)
else:
    trainer = SceneSegTrainer()
```

Practical recommendation:

- Start from scratch (`--checkpoint_path ""`) when your data domain/labels differ significantly.
- Start from checkpoint (`--checkpoint_path /absolute/path/model.pth`) when you want faster convergence
  and more stable early training performance.

#### b. Prepare the training datasets

SceneSeg training data should be generated with scripts under `Models/data_parsing/SceneSeg/` so every dataset is
converted into a unified coarse semantic label format.

Target processed structure for each dataset is:

```json
<DATASET_NAME>/
    |----images/
    |----gt_masks/
```

From repository root (`autoware_vision_pilot/`), run one parser per dataset.

##### i) ACDC

```bash
python3 Models/data_parsing/SceneSeg/ACDC/process_acdc.py \
  -l /path/to/ACDC/labels \
  -i /path/to/ACDC/images \
  -ls /path/to/processed/ACDC/gt_masks/ \
  -is /path/to/processed/ACDC/images/
```

##### ii) MUSES

```bash
python3 Models/data_parsing/SceneSeg/MUSES/process_muses.py \
  -l /path/to/MUSES/labels \
  -i /path/to/MUSES/images \
  -ls /path/to/processed/MUSES/gt_masks/ \
  -is /path/to/processed/MUSES/images/
```

##### iii) IDDAW

`process_iddaw.py` expects label/image files via two-level glob (`*/*`), so provide folders where that structure exists.

```bash
python3 Models/data_parsing/SceneSeg/IDDAW/process_iddaw.py \
  -l /path/to/IDDAW/labels \
  -i /path/to/IDDAW/images \
  -ls /path/to/processed/IDDAW/gt_masks/ \
  -is /path/to/processed/IDDAW/images/
```

##### iv) Mapillary Vistas

`process_mapillary_vistas.py` downsamples by 1/2 and filters invalid samples (`is_valid_image`).
Only valid samples are written.

```bash
python3 Models/data_parsing/SceneSeg/Mapillary_Vistas/process_mapillary_vistas.py \
  -l /path/to/Mapillary_Vistas/labels \
  -i /path/to/Mapillary_Vistas/images \
  -ls /path/to/processed/Mapillary_Vistas/gt_masks/ \
  -is /path/to/processed/Mapillary_Vistas/images/
```

##### v) comma10k

`process_comma10k.py` requires an additional sky-mask input (`-sm`).

```bash
python3 Models/data_parsing/SceneSeg/comma10K/process_comma10k.py \
  -l /path/to/comma10k/labels \
  -sm /path/to/comma10k/sky_masks \
  -i /path/to/comma10k/images \
  -ls /path/to/processed/comma10k/gt_masks/ \
  -is /path/to/processed/comma10k/images/
```

##### vi) BDD100K (optional in current trainer)

Parser is available, but `train_scene_seg.py` does not currently load BDD100K by default.

```bash
python3 Models/data_parsing/SceneSeg/BDD100K/process_bdd100k.py \
  -l /path/to/BDD100K/labels \
  -i /path/to/BDD100K/images \
  -ls /path/to/processed/BDD100K/gt_masks/ \
  -is /path/to/processed/BDD100K/images/
```

**Recommended workflow:**

1. Run each parser into its own processed dataset folder.
2. Verify each processed folder contains matching image/mask counts.
3. Open a few `gt_masks/*.png` files and confirm class colors are sensible.
4. Use these processed folders as inputs for the SceneSeg training script in the next subsection.

Semantic notes (from SceneSeg data-prep README):

- Non-comma10k datasets annotate multiple foreground subclasses (`Vulnerable Living`, `Small Mobile Vehicle`,
  `Large Mobile Vehicle`), which are unified during training into a single foreground concept.
- `comma10k` does not provide `Road Edge Delimiter`; this is treated as background during training.
- `comma10k` requires external sky masks (`-sm`).
- Due to cross-dataset inconsistency, sky is merged with background during training-time class handling.

#### c. How to load dataset

`train_scene_seg.py` loads processed datasets through `LoadDataSceneSeg` using fixed folder names under one root path.
Expected processed tree:

```json
<ROOT_PATH>/
|----ACDC/
|    |----images/
|    |----gt_masks/
|----IDDAW/
|    |----images/
|    |----gt_masks/
|----MUSES/
|    |----images/
|    |----gt_masks/
|----Mapillary_Vistas/
|    |----images/
|    |----gt_masks/
|----comma10k/
|    |----images/
|    |----gt_masks/
```

In current script, these dataset paths are built as string concatenations, e.g.:

```python
acdc_labels_filepath = root + 'ACDC/gt_masks/'
acdc_images_filepath = root + 'ACDC/images/'
```

So ensure:

1. `-r / --root` points to the parent folder containing `ACDC`, `IDDAW`, `MUSES`, `Mapillary_Vistas`, `comma10k`.
2. Root path includes a trailing `/` (because current code concatenates strings directly).

Example training launch with processed data root:

```bash
python3 Models/training/train_scene_seg.py \
  -r /absolute/path/to/processed_root/ \
  -s /absolute/path/to/model_saves/
```

Before full training, run a quick sanity check:

- Confirm every active dataset has both `images/` and `gt_masks/`.
- Confirm counts are non-zero and roughly aligned per dataset.
- Verify a few image/mask pairs visually to ensure preprocessing output looks correct.

Note: BDD100K parser exists, but current `train_scene_seg.py` does not include BDD100K in the active loading block.
To train with BDD100K, add corresponding path declarations and `LoadDataSceneSeg(...)` calls in `train_scene_seg.py`.

#### d. How to run training

After dataset preprocessing is complete and root paths are ready, training is launched with `train_scene_seg.py`.

##### i) Step 1: Prepare paths

`train_scene_seg.py` requires:

- `-r / --root`: processed dataset root (must contain `ACDC`, `IDDAW`, `MUSES`, `Mapillary_Vistas`, `comma10k`).
- `-s / --model_save_root_path`: folder prefix where `.pth` checkpoints are saved.

Example from repository root:

```bash
python3 Models/training/train_scene_seg.py \
  -r /absolute/path/to/processed_root/ \
  -s /absolute/path/to/model_saves/
```

Tip: use trailing `/` on both `-r` and `-s` because current script builds paths with string concatenation.

##### ii) Step 2: What happens during training

- Loads five datasets via `LoadDataSceneSeg` (ACDC, IDDAW, MUSES, MAPILLARY, COMMA10K).
- Iterates mixed-dataset samples each epoch (dataset order is shuffled once per epoch).
- Applies segmentation augmentations (`Augmentations(..., data_type='SEGMENTATION')`).
- Runs forward + weighted `CrossEntropyLoss` and accumulates gradients.
- Performs optimizer step according to dynamic gradient-accumulation batch schedule:
  - epoch 0: 32, epoch 1: 16, epoch 2: 8, epoch 3: 5, epoch 4-5: 3, epoch 6-7: 2, epoch 9+: 1.

##### iii) Step 3: Logging / checkpoint / validation cadence

- Every 250 steps: logs training loss to TensorBoard (`Loss/train`).
- Every 1000 steps: logs prediction-vs-ground-truth visualization figure to TensorBoard.
- Every 8000 steps:
  - saves checkpoint `.pth` to `model_save_root_path`,
  - runs full validation over all active datasets,
  - logs `Val/IoU` and per-class IoU (`mIoU_bg`, `mIoU_fg`, `mIoU_rd`) to TensorBoard.

Training finishes by flushing and closing TensorBoard writer (`trainer.cleanup()`).

#### e. How to visualize training results

This training pipeline logs metrics and qualitative outputs primarily through **TensorBoard**.

##### a) Open TensorBoard

From project root, run:

```bash
tensorboard --logdir runs --port 6006
```

Then open `http://localhost:6006` in your browser.

##### b) What you should monitor

- Scalars:

  - `Loss/train` (logged every 250 training steps)
  - `Val/IoU` (overall mean IoU, logged every 8000 steps)
  - `Val/IoU_Classes` (`mIoU_bg`, `mIoU_fg`, `mIoU_rd`)
  
- Images/Figures:
  - `predictions vs. actuals` panel (logged every 1000 steps), showing
    input image, ground truth, and model prediction side-by-side.

##### c) Checkpoints and practical notes

- Checkpoints are saved every 8000 steps to the path passed via `-s / --model_save_root_path`.
- Current SceneSeg trainer does **not** save standalone PNG validation figures to disk by default; visualization is
  provided via TensorBoard figure logging.
- If you need more frequent monitoring during debug runs, reduce logging intervals in `train_scene_seg.py`
  (e.g., 250/1000/8000 thresholds).

### 2. Building an Inference Pipeline

#### a. Using ROS2 to publish an image, run inference and visualize results

TBD.

#### b. Using IceOryx to publish an image, run inference and visualize results

TBD.

#### c. Using Zenoh to publish an image, run inference and visualize results

TBD.

#### d. Using Gstreamer to get an image, run inference and visualize results

TBD.

## IV. Lite version - SceneSegLite

### 1. Lite version introduction

SceneSegLite is a lightweight semantic segmentation network designed for
efficient deployment on embedded edge devices. It performs a somewhat
**Cityscapes-style semantic classification** with 19 standard classes while
maintaining dramatically reduced computational overhead compared to the
original SceneSeg model.

#### a. Key architecture differences

**SceneSegLite vs. Original SceneSeg:**

| Aspect | SceneSeg | SceneSegLite |
|--------|----------|--------------|
| **Encoder** | EfficientNet-B0 | EfficientNet-B1 |
| **Decoder** | Full Scene Neck | Lightweight DeepLabV3+ |
| **Classes** | 3 (background, foreground, road) | 19 (Cityscapes standard) |
| **Compute (GOPs)** | 224 | 7.82 |
| **FPS (INT8 TensorRT)** | ~10 | 87.6 |
| **Device** | GPU-dominant | Jetson Orin Nano (8GB) |

#### b. The 19 Cityscapes classes

SceneSegLite performs fine-grained semantic segmentation grouped into the following categories:

- Flat surfaces: road, sidewalk  
- Construction: building, wall, fence  
- Objects: pole, traffic light, traffic sign  
- Nature: vegetation, terrain  
- Sky  
- Humans: pedestrian, vehicle rider  
- Vehicles: car, truck, bus, train, motorcycle, bicycle

#### c. Performance metrics (Jetson Orin Nano 8GB)

| Model | GOPs | Backend | Forward [ms] | FPS |
|-------|------|---------|--------------|-----|
| SceneSeg | 224 | FP32 ONNX | 159.6 | 6.3 |
| SceneSeg | 224 | FP32 TensorRT | 98.1 | 10.2 |
| SceneSegLite | 7.82 | FP32 ONNX | 43.5 | 23.0 |
| SceneSegLite | 7.82 | FP32 TensorRT | 23.9 | 41.8 |
| SceneSegLite | 7.82 | INT8 TensorRT | 11.4 | 87.6 |

The "forward pass" includes Host => Device (H2D) transfer + inference + Device => Host (D2H) transfer.

#### d. When to use SceneSegLite

- Real-time urban scene understanding on resource-constrained vehicles.
- Multi-camera deployments requiring parallel perception streams.
- Fine-grained semantic reasoning (19 classes vs. 3 coarse classes in SceneSeg).
- Edge computing scenarios with strict power and latency budgets.
- Production autonomous systems requiring INT8-quantized inference.

### 2. Lite version model weights download

The SceneSegLite model weights are available in both PyTorch and ONNX formats. 
You can manually download them using the links below, or use the automated 
Python script provided in this section.

Pre-trained weights should be placed in the directory: `autoware_vision_pilot/Tutorials/E2E_Models/weights/`.

- [Link to download PyTorch weights (*.pth)](https://drive.google.com/file/d/1nxOg-Bl5f5PVm90YebMUnu-wKWoDXG-n/view?usp=drive_link).
- [Link to download ONNX FP32 (*.onnx)](https://drive.google.com/file/d/1MqDNmaHz354dH9xQ2W5NT9V7P8rE9HgA/view?usp=drive_link).

The following script uses `gdown` to download the weights automatically into the default directory.

In [ ]:
import gdown

model_dir = "./weights"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

metadata_lite = {
    "pth"   : {
        "url"       : "https://drive.google.com/uc?id=1nxOg-Bl5f5PVm90YebMUnu-wKWoDXG-n",
        "filepath"  : os.path.join(base_dir, model_dir, "sceneseg_lite.pth")
    },
    "onnx"  : {
        "url"       : "https://drive.google.com/uc?id=1MqDNmaHz354dH9xQ2W5NT9V7P8rE9HgA",
        "filepath"  : os.path.join(base_dir, model_dir, "sceneseg_lite.onnx")
    }
}

for weight_type in metadata_lite:

    url         = metadata_lite[weight_type]["url"]
    filepath    = metadata_lite[weight_type]["filepath"]

    if not os.path.exists(filepath):
        print(f"Downloading SceneSegLite {weight_type.upper()} weights...")
        gdown.download(url, filepath, quiet = False)
    else:
        print(f"SceneSegLite {weight_type.upper()} weights already exist at {filepath}. Skipping download.")

### 3. Image Inference

SceneSegLite supports the same inference workflow as the original SceneSeg, with identical visualization scripts. The key improvement is the significantly faster inference speed and the fine-grained 19-class semantic output.

#### a. Run Batch Image Inference

The same visualization scripts work for SceneSegLite. Simply provide the path to the SceneSegLite checkpoint instead.

In [ ]:
# 1. Path declaration for SceneSegLite
CHECKPOINT_LITE = metadata_lite["pth"]["filepath"]
INPUT_DIR_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/images/"
)
OUTPUT_DIR_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/sceneseg_lite/images/"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/SceneSeg && \
python3 image_visualization.py \
    -p {os.path.abspath(CHECKPOINT_LITE)} \
    -i {os.path.abspath(INPUT_DIR_LITE)} \
    -o {os.path.abspath(OUTPUT_DIR_LITE)}

#### b. Display Results in Notebook

In [ ]:
# Display SceneSegLite inference results
result_files_lite = sorted([
    f for f in os.listdir(OUTPUT_DIR_LITE)
    if f.endswith(".png")
])

if (result_files_lite):
    # Display the first 3 results
    fig, axes = plt.subplots(
        1, 3,
        figsize = (20, 10)
    )
    for i, ax in enumerate(axes):
        if (i < len(result_files_lite)):
            img_path = os.path.join(OUTPUT_DIR_LITE, result_files_lite[i])
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"SceneSegLite Result: {result_files_lite[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No result images found. Check your output directory.")

### 4. Video Inference

The SceneSegLite model can be applied to video streams in the same way as the 
original SceneSeg. The `video_visualization.py` script processes each frame 
with SceneSegLite's efficient inference engine, producing detailed 19-class 
semantic segmentation overlays.

Key advantages when processing video with SceneSegLite:
- High throughput video processing capacity, at 87+ FPS on embedded hardware with INT8 quantization.
- Single-frame low latency under 12ms on Jetson platforms.
- Rich semantic detail featuring 19 fine-grained classes provide comprehensive scene understanding.
- Optimized for continuous real-time operation on edge devices.

In [ ]:
# 1. Path declaration for SceneSegLite video inference
INPUT_VIDEO_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/assets/videos/highway_normal.mp4"
)
OUTPUT_VIDEO_LITE = os.path.join(
    base_dir,
    "./autoware_vision_pilot/Tutorials/E2E_Models/results/sceneseg_lite/videos/highway_normal_output.avi"
)

# 2. Navigate to visualization directory and execute the script
!cd autoware_vision_pilot/Models/visualizations/SceneSeg && \
python3 video_visualization.py \
    -p {os.path.abspath(CHECKPOINT_LITE)} \
    -i {os.path.abspath(INPUT_VIDEO_LITE)} \
    -o {os.path.abspath(OUTPUT_VIDEO_LITE)}